In [5]:
import torch
import torch.nn as nn
import math

class GroupedQueryAttention(nn.Module):
    def __init__(self, hidden_size, num_heads, kv_groups=1):
        super(GroupedQueryAttention, self).__init__()
        
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.kv_groups = kv_groups
        self.head_dim = hidden_size // num_heads
        
        # 计算 KV 的头数
        self.num_kv_heads = num_heads // kv_groups
        
        assert hidden_size % num_heads == 0, "hidden_size must be divisible by num_heads"
        assert num_heads % kv_groups == 0, "num_heads must be divisible by kv_groups"

        # Q 保持全量
        self.w_q = nn.Linear(hidden_size, hidden_size)
        
        # K, V 维度减少
        self.w_k = nn.Linear(hidden_size, self.num_kv_heads * self.head_dim)
        self.w_v = nn.Linear(hidden_size, self.num_kv_heads * self.head_dim)
        
        self.w_o = nn.Linear(hidden_size, hidden_size)

    def repeat_kv(self, x, kv_groups):
        """
        将 KV head 沿着 dim=2 扩充 kv_groups 倍。
        x: (batch, num_kv_heads, seq_len, head_dim)
        kv_groups: 复制倍数
        return: (batch, num_kv_heads * kv_groups, seq_len, head_dim)
        """
        batch_size, num_kv_heads, seq_len, head_dim = x.shape
        
        if kv_groups == 1:
            return x
            
        # 1. Unsqueeze: 插入 group 维度
        # (B, H_kv, L, D) -> (B, H_kv, 1, L, D)
        x = x.unsqueeze(2)
        
        # 2. Expand: 广播数据 (内存高效)
        # (B, H_kv, 1, L, D) -> (B, H_kv, kv_groups, L, D)
        x = x.expand(batch_size, num_kv_heads, kv_groups, seq_len, head_dim)
        
        # 3. Reshape: 合并头维度
        # (B, H_kv, kv_groups, L, D) -> (B, H_kv * kv_groups, L, D)
        x = x.reshape(batch_size, num_kv_heads * kv_groups, seq_len, head_dim)
        
        return x

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        
        # 1. 投影 + 分头 + Transpose
        # Q: (B, H_q, L, D)
        Q = self.w_q(q).view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        
        # K, V: (B, H_kv, L, D)
        K = self.w_k(k).view(batch_size, -1, self.num_kv_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(v).view(batch_size, -1, self.num_kv_heads, self.head_dim).transpose(1, 2)

        # 2. GQA 核心：调用 repeat_kv 补全维度
        # 将 K, V 从 (B, H_kv, L, D) 扩展为 (B, H_q, L, D)
        if self.kv_groups > 1:
            K = self.repeat_kv(K, self.kv_groups)
            V = self.repeat_kv(V, self.kv_groups)

        # 3. 注意力计算 (此时维度已对齐)
        # scores: (B, H_q, L, L)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attn = torch.softmax(scores, dim=-1)
        
        # context: (B, H_q, L, D)
        context = torch.matmul(attn, V)

        # 4. 输出投影
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.hidden_size)
        return self.w_o(context)

In [6]:
# 配置验证
B, L, D = 2, 10, 64
H = 8
GROUPS = 4

model = GroupedQueryAttention(D, H, kv_groups=GROUPS)
x = torch.randn(B, L, D)

out = model(x, x, x)

print(f"Num KV Heads: {model.num_kv_heads}") # 应该是 2
print(f"Output Shape: {out.shape}")           # 应该是 (2, 10, 64)
print("✅ Logic Verified")

Num KV Heads: 2
Output Shape: torch.Size([2, 10, 64])
✅ Logic Verified
